In [ ]:
!pip --quiet install -U langchain langchain-core==1.0 langchain-community langchain-experimental langchain_google_genai chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0

In [ ]:
# Upgrade langchain-core to resolve dependency conflicts with langchain-openai and ragas
!pip --quiet install -U "langchain-core>=1.7.5"

After running the pip installation above, you may need to **restart the session** (Runtime -> Restart session) for Python to register the newly installed package version, and then execute your imports again.

### 0. Dataset and File Preparation for Education

Let's programmatically write two highly structured text files representing documents with distinct themes (e.g., Space Exploration and Deep Sea Biology). This provides the clear topical variance needed to easily demonstrate **Metadata Filtering**, **Parent Document Retrieval**, and **Hybrid Search** with complete context.

In [ ]:
import os

# Create a couple of structured educational text files
space_content = """Space Exploration: The Journey to Mars

Human exploration of Mars remains a grand ambition of the 21st century. Planning a mission to Mars requires solving complex challenges in rocket propulsion, life support systems, and radiation shielding.

Historically, robotic rovers like Curiosity and Perseverance have explored Martian craters, analyzing soil samples for signs of ancient microbial life. The atmosphere of Mars is thin, composed of over 95% carbon dioxide, which presents extreme temperature shifts and makes breathing impossible without life-support technology.

Developing sustained colonies will rely on resource utilization (ISRU) to turn Martian water ice and atmospheric gases into fuel, oxygen, and water. Scientists hope to land humans on Mars by the late 2030s."""

ocean_content = """Deep Sea Exploration: Mysteries of the Mariana Trench

The Mariana Trench is the deepest known point on Earth, plunging nearly 11 kilometers (36,000 feet) down in the Western Pacific Ocean. At this depth, the pressure is immense, exceeding 1,000 atmospheres.

Despite the extreme environment, life still thrives here. Specialized organisms such as amphipods, snailfish, and xenophyophores have adapted to survive in absolute darkness and near-freezing temperatures.

Explorers use remotely operated vehicles (ROVs) and manned submersibles to map the sea floor and harvest geological and biological samples. Oceanographers emphasize that studying these deep-sea ecosystems helps us understand the origins of life on Earth."""

with open('space.txt', 'w', encoding='utf-8') as f:
    f.write(space_content)

with open('ocean.txt', 'w', encoding='utf-8') as f:
    f.write(ocean_content)

print("Files 'space.txt' and 'ocean.txt' successfully created for notebook demonstration!")

Files 'space.txt' and 'ocean.txt' successfully created for notebook demonstration!


### Initial Loading and Setup
Let's load these documents, split them, and initialize our primary embeddings and Chroma vector store so the subsequent steps run cleanly.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata

# 1. Load the generated documents
loaders = [
    TextLoader('space.txt'),
    TextLoader('ocean.txt')
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

# 2. Chunk documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
final_documents = text_splitter.split_documents(docs)

# 3. Setup embeddings (using Gemini SDK with secrets API key)
api_key = userdata.get('gemini_key')
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=api_key)

# 4. Build base Vector Store
db = Chroma.from_documents(final_documents, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 2})

# Build LLM for RAG
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", google_api_key=api_key)

print(f"Ingested {len(final_documents)} chunks into Chroma.")

Ingested 7 chunks into Chroma.


### Metadata Filtering with Chroma

Metadata filtering helps you narrow down search results to specific documents matching particular attributes (e.g., matching the `source` filename) before performing similarity search. This improves both retrieval speed and accuracy.

In [ ]:
# === Metadata Filtering Example ===
# Let's search the vector store specifically filtering by document source.
# This ensures that even if both files have similar words, we only get results from our specified document.

# 1. Target the space exploration file specifically
space_filter = {"source": "space.txt"}

# 2. Query matching terms that exist in both themes (such as 'exploration' or 'environment')
# but specify the filter targeting only 'space.txt'
space_query_results = db.similarity_search(
    query="What challenges are faced during exploration?",
    k=2,
    filter=space_filter
)

print("=== Metadata Filter Results (source: space.txt) ===")
for i, doc in enumerate(space_query_results):
    print(f"\nChunk {i+1}:")
    print(f"Source File: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content}")

# 3. Repeat the search targeting 'ocean.txt' to show how the filter alters the scope
ocean_filter = {"source": "ocean.txt"}
ocean_query_results = db.similarity_search(
    query="What challenges are faced during exploration?",
    k=2,
    filter=ocean_filter
)

print("\n=== Metadata Filter Results (source: ocean.txt) ===")
for i, doc in enumerate(ocean_query_results):
    print(f"\nChunk {i+1}:")
    print(f"Source File: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content}")

=== Metadata Filter Results (source: space.txt) ===

Chunk 1:
Source File: space.txt
Content: Space Exploration: The Journey to Mars

Human exploration of Mars remains a grand ambition of the 21st century. Planning a mission to Mars requires solving complex challenges in rocket propulsion, life support systems, and radiation shielding.

Chunk 2:
Source File: space.txt
Content: temperature shifts and makes breathing impossible without life-support technology.

=== Metadata Filter Results (source: ocean.txt) ===

Chunk 1:
Source File: ocean.txt
Content: Explorers use remotely operated vehicles (ROVs) and manned submersibles to map the sea floor and harvest geological and biological samples. Oceanographers emphasize that studying these deep-sea ecosystems helps us understand the origins of life on Earth.

Chunk 2:
Source File: ocean.txt
Content: Deep Sea Exploration: Mysteries of the Mariana Trench

The Mariana Trench is the deepest known point on Earth, plunging nearly 11 kilometers (36,

### Advanced RAG Techniques: Grounding, Optimization, and Hybrid Search

This section covers industry best practices to improve Retrieval-Augmented Generation (RAG) performance:
1. **Grounding & Guardrails**: Restricting the LLM to only answer from context and implementing verification.
2. **Query & Chunk Optimization**: Using techniques like Parent Document Retrieval and query transformation.
3. **Hybrid Search & Reranking**: Combining keyword (BM25) and dense vector search, then ranking results with a cross-encoder model.

#### 1. Query & Chunk Optimization (Parent Document Retriever)
Instead of splitting documents into small chunks and losing context, we can index smaller chunks for retrieval, but return their larger parent documents to the LLM to preserve complete context.

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Define splitters: Child splitter for vector storage, Parent splitter
#for context retrieval
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

# 2. Setup the storage layer for full parent documents
store = InMemoryStore()

# 3. Initialize ParentDocumentRetriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=db,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Add the initial loaded document as a demonstration
parent_retriever.add_documents(docs, ids=None)
print("Parent-Child Retriever configured and populated.")

Parent-Child Retriever configured and populated.


In [ ]:
# === Child Retriever vs. Parent Document Retriever Comparison ===
# This cell demonstrates how the child retriever fetches small, isolated chunks
# while the Parent Document Retriever restores the full surrounding context.

query = "What is the atmosphere of Mars composed of?"

# 1. Retrieve using the Child Vector Store Retriever directly
# This shows what a standard vector database search returns when splitting text into small chunks
child_results = db.similarity_search(query, k=1)
print("=== Child Retriever (Direct Vector Store Search) ===")
for doc in child_results:
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Snippet Length: {len(doc.page_content)} characters")
    print(f"Snippet Content: \n{doc.page_content}\n")

# 2. Retrieve using the Parent Document Retriever
# This shows how the same query maps back to the larger parent chunk to provide complete context to the LLM
parent_results = parent_retriever.invoke(query)
print("=== Parent Document Retriever ===")
for doc in parent_results:
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Full Context Length: {len(doc.page_content)} characters")
    print(f"Full Context Content: \n{doc.page_content}")

=== Child Retriever (Direct Vector Store Search) ===
Source: space.txt
Snippet Length: 326 characters
Snippet Content: 
Historically, robotic rovers like Curiosity and Perseverance have explored Martian craters, analyzing soil samples for signs of ancient microbial life. The atmosphere of Mars is thin, composed of over 95% carbon dioxide, which presents extreme temperature shifts and makes breathing impossible without life-support technology.

=== Parent Document Retriever ===
Source: space.txt
Full Context Length: 779 characters
Full Context Content: 
Space Exploration: The Journey to Mars

Human exploration of Mars remains a grand ambition of the 21st century. Planning a mission to Mars requires solving complex challenges in rocket propulsion, life support systems, and radiation shielding.

Historically, robotic rovers like Curiosity and Perseverance have explored Martian craters, analyzing soil samples for signs of ancient microbial life. The atmosphere of Mars is thin, composed of 

**Explanation:** Under the hood, the ParentDocumentRetriever uses your vector store (db) to index and search the smaller child chunks (created by child_splitter).

When you call parent_retriever.invoke(query), the retriever first executes a standard similarity search on the vector database (db) using the child retriever to find the most relevant tiny child chunks.
2. How is it covered by the parent retriever?
Once the child retriever identifies the best small chunks (e.g., "The atmosphere of Mars is thin, composed of over 95% carbon..."), the parent retriever takes over:

It looks at the metadata of those retrieved child chunks to find the unique ID of their corresponding parent document.
It queries the in-memory document store (store) using that ID.
It fetches and returns the full parent document (the entire space.txt content) instead of the small snippet.
By doing this, the parent retriever combines the best of both worlds: efficient, precise search (using small child chunks) and complete context delivery (by feeding the full parent document to your LLM).

#### 2. Hybrid Search and Reranking
Hybrid search combines sparse representations (BM25 keyword search) with dense representations (Vector Search). We then use a Cohere or Flashrank-based reranking step to sort the combined results.

In [ ]:
!pip install rank_bm25

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# 1. Initialize BM25 Keyword Retriever
bm25_retriever = BM25Retriever.from_documents(final_documents)
bm25_retriever.k = 2

# 2. Combine Dense (Chroma) and Sparse (BM25) search using EnsembleRetriever with relative weights
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

print("Hybrid Search (Ensemble Retriever) established.")

Hybrid Search (Ensemble Retriever) established.


In [ ]:
# === Hybrid Search Demonstration ===
# This demonstration shows how sparse (BM25) and dense (Chroma vector store)
# search retrievers are combined to execute hybrid queries on our corpus.

# 1. Define a search query that contains both explicit terms (great for BM25) and conceptual meanings (great for Vector Search)
query = "Mariana Trench depth and extreme pressure"

# 2. Retrieve documents from the individual retrievers to see how they differ
print("--- Sparse Retrieval (BM25 Keyword Search) Results ---")
bm25_docs = bm25_retriever.invoke(query)
for doc in bm25_docs:
    print(f"Source: {doc.metadata.get('source')} | Text: {doc.page_content[:100]}...")

print("\n--- Dense Retrieval (Semantic Vector Search) Results ---")
vector_docs = retriever.invoke(query)
for doc in vector_docs:
    print(f"Source: {doc.metadata.get('source')} | Text: {doc.page_content[:100]}...")

# 3. Retrieve using the Ensemble (Hybrid) Retriever
print("\n--- Hybrid Retrieval (Ensemble Retriever) Combined Results ---")
hybrid_docs = ensemble_retriever.invoke(query)
for i, doc in enumerate(hybrid_docs):
    print(f"Rank {i+1} | Source: {doc.metadata.get('source')} | Content: {doc.page_content[:150]}...")

--- Sparse Retrieval (BM25 Keyword Search) Results ---
Source: ocean.txt | Text: Deep Sea Exploration: Mysteries of the Mariana Trench

The Mariana Trench is the deepest known point...
Source: ocean.txt | Text: Despite the extreme environment, life still thrives here. Specialized organisms such as amphipods, s...

--- Dense Retrieval (Semantic Vector Search) Results ---
Source: ocean.txt | Text: Deep Sea Exploration: Mysteries of the Mariana Trench

The Mariana Trench is the deepest known point...
Source: ocean.txt | Text: Deep Sea Exploration: Mysteries of the Mariana Trench

The Mariana Trench is the deepest known point...

--- Hybrid Retrieval (Ensemble Retriever) Combined Results ---
Rank 1 | Source: ocean.txt | Content: Deep Sea Exploration: Mysteries of the Mariana Trench

The Mariana Trench is the deepest known point on Earth, plunging nearly 11 kilometers (36,000 f...
Rank 2 | Source: ocean.txt | Content: Despite the extreme environment, life still thrives here. Specialized org

#### 3. Grounding Techniques and Guardrails
To guarantee safety and prevent hallucinations, we can apply structural system prompt constraints (Grounding) and programmatically parse the response to enforce rules (Guardrails).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# 1. A strict prompt template enforcing grounding boundaries
grounded_prompt = ChatPromptTemplate.from_template(
    """You are an extremely strict AI assistant.

Instructions:
- Answer the user's question using ONLY the facts explicitly mentioned in the context below.
- If the context does not contain the answer, reply exactly with: "I cannot answer this based on the provided context."
- Do NOT extrapolate, assume, or use external knowledge.

Context:
{context}

Question: {input}

Answer:"""
)

# 2. Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Constructing the complete grounded RAG chain using the Hybrid Ensemble Retriever
grounded_rag_chain = (
    RunnableParallel({"context": ensemble_retriever | format_docs, "input": RunnablePassthrough()})
    | grounded_prompt
    | llm
    | StrOutputParser()
)

# Test the grounded chain with an in-context query about Mars rovers
in_context_query = "Which robotic rovers have explored Martian craters?"
print(f"--- Grounded RAG Response (In-Context Query) ---")
print(grounded_rag_chain.invoke(in_context_query))

# Test the grounded chain with an out-of-context query to check guardrails
out_of_context_query = "Who painted the Mona Lisa?"
print(f"\n--- Grounded RAG Response (Out-of-Context Query Guardrail Check) ---")
print(grounded_rag_chain.invoke(out_of_context_query))

--- Grounded RAG Response (In-Context Query) ---
Curiosity and Perseverance

--- Grounded RAG Response (Out-of-Context Query Guardrail Check) ---
I cannot answer this based on the provided context.


In [ ]:
# === Grounding & Guardrails Demonstration ===
# This demonstration evaluates our RAG setup to verify that it successfully
# utilizes context when available, but accurately triggers guardrails for out-of-bounds questions.

# Test 1: Query clearly present in our space text document
query_in_context = "What is the main atmospheric gas on Mars and what percentage of it is there?"
print(f"[Query 1]: {query_in_context}")
try:
    response_1 = grounded_rag_chain.invoke(query_in_context)
    print(f"[Answer 1]: {response_1}\n")
except Exception as e:
    print(f"Error: {e}\n")

# Test 2: Query clearly present in our ocean text document
query_ocean_context = "What specialized organisms adapt to survive in the Mariana Trench?"
print(f"[Query 2]: {query_ocean_context}")
try:
    response_2 = grounded_rag_chain.invoke(query_ocean_context)
    print(f"[Answer 2]: {response_2}\n")
except Exception as e:
    print(f"Error: {e}\n")

# Test 3: Completely out-of-context query to verify prompt guardrails block hallucinations
query_out_of_bounds = "What are the main ingredients of a standard Margherita pizza?"
print(f"[Query 3 (Out-of-Bounds)]: {query_out_of_bounds}")
try:
    response_3 = grounded_rag_chain.invoke(query_out_of_bounds)
    print(f"[Answer 3]: {response_3}")
except Exception as e:
    print(f"Error: {e}")

[Query 1]: What is the main atmospheric gas on Mars and what percentage of it is there?
[Answer 1]: Carbon dioxide, and it is composed of over 95%.

[Query 2]: What specialized organisms adapt to survive in the Mariana Trench?
[Answer 2]: I cannot answer this based on the provided context.

[Query 3 (Out-of-Bounds)]: What are the main ingredients of a standard Margherita pizza?
[Answer 3]: I cannot answer this based on the provided context.
